In [ ]:
"""
TEST NOTEBOOK - PHUONG PHAP 3 (VIT_HYBRID).
Tai su dung pipeline danh gia cua test_v2 nhung thay model bang VIT_HYBRID (models_v3).
"""
from ruamel.yaml import YAML
import numpy as np

import torch
import torch.backends.cudnn as cudnn
import torch.nn.functional as F

from transformers import BertTokenizerFast
from MMDF_.dataset import create_dataset, create_loader
from tqdm import tqdm

from sklearn.metrics import roc_auc_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

from MMDF_.models_v3 import VIT_HYBRID
from MMDF_.utils import text_input_adjust, AttrDict

from MMDF_.models import box_ops
from MMDF_.tools.multilabel_metrics import AveragePrecisionMeter, get_multi_label

In [ ]:
args = {
    'config': r'MMDF_\configs\test_v2.yaml',
    'checkpoint': r'checkpoints_v3\vit_hybrid_01.pth',
    'resume': False,
    'output_dir': 'results_v3',
    'text_encoder': 'bert-base-uncased',
    'device': 'cuda',
    'world_size': 1,
    'launcher': 'pytorch',
    'model_save_epoch': 5,
    'token_momentum': False,
    'test_epoch': 'best',
}
args = AttrDict(args)
yaml = YAML(typ='safe')
with open(args.config, 'r') as f:
    config = yaml.load(f)

In [ ]:
@torch.no_grad()
def evaluation(args, model, data_loader, tokenizer, device, config):
    model.eval()
    print('[v3] Computing features for evaluation...')

    y_true, y_pred, IOU_pred, IOU_50, IOU_75, IOU_95 = [], [], [], [], [], []
    cls_nums_all, cls_acc_all = 0, 0

    TP_all = TN_all = FP_all = FN_all = 0
    TP_all_multicls = np.zeros(4, dtype=int)
    TN_all_multicls = np.zeros(4, dtype=int)
    FP_all_multicls = np.zeros(4, dtype=int)
    FN_all_multicls = np.zeros(4, dtype=int)
    F1_multicls = np.zeros(4)

    multi_label_meter = AveragePrecisionMeter(difficult_examples=False)
    multi_label_meter.reset()

    for image, label, text, fake_image_box, fake_word_pos, W, H in tqdm(data_loader):
        image = image.to(device, non_blocking=True)
        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True,
                               return_attention_mask=True, return_token_type_ids=False)
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device)

        logits_real_fake, logits_multicls, output_coord, logits_tok = model(
            image, label, text_input, fake_image_box, fake_token_pos, is_train=False
        )

        cls_label = torch.ones(len(label), dtype=torch.long, device=image.device)
        real_pos = np.where(np.array(label) == 'orig')[0].tolist()
        cls_label[real_pos] = 0
        y_pred.extend(F.softmax(logits_real_fake, dim=1)[:, 1].cpu().tolist())
        y_true.extend(cls_label.cpu().tolist())
        cls_acc_all += (logits_real_fake.argmax(1) == cls_label).sum().item()
        cls_nums_all += cls_label.size(0)

        target, _ = get_multi_label(label, image)
        multi_label_meter.add(logits_multicls, target)
        for cls_idx in range(logits_multicls.shape[1]):
            cls_pred = logits_multicls[:, cls_idx].clone()
            cls_pred[cls_pred >= 0] = 1
            cls_pred[cls_pred < 0] = 0
            TP_all_multicls[cls_idx] += ((target[:, cls_idx] == 1) * (cls_pred == 1)).sum().item()
            TN_all_multicls[cls_idx] += ((target[:, cls_idx] == 0) * (cls_pred == 0)).sum().item()
            FP_all_multicls[cls_idx] += ((target[:, cls_idx] == 0) * (cls_pred == 1)).sum().item()
            FN_all_multicls[cls_idx] += ((target[:, cls_idx] == 1) * (cls_pred == 0)).sum().item()

        b1 = box_ops.box_cxcywh_to_xyxy(output_coord)
        b2 = box_ops.box_cxcywh_to_xyxy(fake_image_box.to(device))
        IOU, _ = box_ops.box_iou(b1, b2, test=True)
        IOU_pred.extend(IOU.cpu().tolist())
        IOU_50.extend((IOU > 0.5).long().cpu().tolist())
        IOU_75.extend((IOU > 0.75).long().cpu().tolist())
        IOU_95.extend((IOU > 0.95).long().cpu().tolist())

        token_label = text_input.attention_mask[:, 1:].clone()
        token_label[token_label == 0] = -100
        token_label[token_label == 1] = 0
        for b_idx, positions in enumerate(fake_token_pos):
            for pos in positions:
                if pos < token_label.size(1):
                    token_label[b_idx, pos] = 1
        logits_tok_aligned = logits_tok[:, 1:, :]
        L = min(logits_tok_aligned.size(1), token_label.size(1))
        pred_tok = logits_tok_aligned[:, :L, :].argmax(-1).reshape(-1)
        tlab = token_label[:, :L].reshape(-1)
        mask_valid = tlab != -100
        pred_tok = pred_tok[mask_valid]
        tlab = tlab[mask_valid]
        TP_all += ((tlab == 1) * (pred_tok == 1)).sum().item()
        TN_all += ((tlab == 0) * (pred_tok == 0)).sum().item()
        FP_all += ((tlab == 0) * (pred_tok == 1)).sum().item()
        FN_all += ((tlab == 1) * (pred_tok == 0)).sum().item()

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    AUC_cls = roc_auc_score(y_true, y_pred)
    ACC_cls = cls_acc_all / cls_nums_all
    fpr, tpr, _ = roc_curve(y_true, y_pred, pos_label=1)
    EER_cls = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

    IOU_score = sum(IOU_pred) / len(IOU_pred)
    IOU_ACC_50 = sum(IOU_50) / len(IOU_50)
    IOU_ACC_75 = sum(IOU_75) / len(IOU_75)
    IOU_ACC_95 = sum(IOU_95) / len(IOU_95)

    ACC_tok = (TP_all + TN_all) / max(TP_all + TN_all + FP_all + FN_all, 1)
    Precision_tok = TP_all / max(TP_all + FP_all, 1)
    Recall_tok = TP_all / max(TP_all + FN_all, 1)
    F1_tok = 2 * Precision_tok * Recall_tok / max(Precision_tok + Recall_tok, 1e-8)

    MAP = multi_label_meter.value().mean()
    OP, OR, OF1, CP, CR, CF1 = multi_label_meter.overall()
    for c in range(4):
        P = TP_all_multicls[c] / max(TP_all_multicls[c] + FP_all_multicls[c], 1)
        R = TP_all_multicls[c] / max(TP_all_multicls[c] + FN_all_multicls[c], 1)
        F1_multicls[c] = 2 * P * R / max(P + R, 1e-8)

    return (AUC_cls, ACC_cls, EER_cls,
            MAP.item(), OP, OR, OF1, CP, CR, CF1, F1_multicls,
            IOU_score, IOU_ACC_50, IOU_ACC_75, IOU_ACC_95,
            ACC_tok, Precision_tok, Recall_tok, F1_tok)

In [ ]:
def main_worker(args, config):
    device = torch.device(args.device)
    cudnn.benchmark = True

    tokenizer = BertTokenizerFast.from_pretrained(args.text_encoder)
    model = VIT_HYBRID(args=args, config=config, text_encoder=args.text_encoder,
                       tokenizer=tokenizer, init_deit=True)
    model = model.to(device)

    if args.checkpoint:
        ckpt = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
        msg = model.load_state_dict(ckpt['model'], strict=False)
        print('[v3] load ckpt:', msg)

    _, val_dataset = create_dataset(config)
    val_loader = create_loader(
        [val_dataset],
        batch_size=[config['batch_size_val']],
        num_workers=[4],
        is_trains=[False],
    )[0]

    (AUC_cls, ACC_cls, EER_cls,
     MAP, OP, OR, OF1, CP, CR, CF1, F1_multicls,
     IOU_score, IOU_ACC_50, IOU_ACC_75, IOU_ACC_95,
     ACC_tok, Precision_tok, Recall_tok, F1_tok) = evaluation(
        args, model, val_loader, tokenizer, device, config
    )

    val_stats = {
        'AUC_cls': f'{AUC_cls*100:.4f}',
        'ACC_cls': f'{ACC_cls*100:.4f}',
        'EER_cls': f'{EER_cls*100:.4f}',
        'MAP': f'{MAP*100:.4f}',
        'OP': f'{OP*100:.4f}', 'OR': f'{OR*100:.4f}', 'OF1': f'{OF1*100:.4f}',
        'CP': f'{CP*100:.4f}', 'CR': f'{CR*100:.4f}', 'CF1': f'{CF1*100:.4f}',
        'F1_FS': f'{F1_multicls[0]*100:.4f}',
        'F1_FA': f'{F1_multicls[1]*100:.4f}',
        'F1_TS': f'{F1_multicls[2]*100:.4f}',
        'F1_TA': f'{F1_multicls[3]*100:.4f}',
        'IOU_score': f'{IOU_score*100:.4f}',
        'IOU_ACC_50': f'{IOU_ACC_50*100:.4f}',
        'IOU_ACC_75': f'{IOU_ACC_75*100:.4f}',
        'IOU_ACC_95': f'{IOU_ACC_95*100:.4f}',
        'ACC_tok': f'{ACC_tok*100:.4f}',
        'Precision_tok': f'{Precision_tok*100:.4f}',
        'Recall_tok': f'{Recall_tok*100:.4f}',
        'F1_tok': f'{F1_tok*100:.4f}',
    }
    return val_stats

In [ ]:
main_worker(args, config)
